<a href="https://colab.research.google.com/github/google-deepmind/hybrid_rnns_reward_learning/blob/main/hybrid_rnns_reward_learning/train_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installs, imports, etc.

In [1]:
#@title Install Hybrid RNNs repo from github
# !rm -r hybrid_rnns_reward_learning
!git clone https://github.com/google-deepmind/hybrid_rnns_reward_learning
%cd hybrid_rnns_reward_learning
!pip install .
%cd ..

Cloning into 'hybrid_rnns_reward_learning'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 86 (delta 46), reused 18 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 35.48 KiB | 1.42 MiB/s, done.
Resolving deltas: 100% (46/46), done.
/content/hybrid_rnns_reward_learning
Processing /content/hybrid_rnns_reward_learning
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hybrid_rnns_reward_learning: filename=hybrid_rnns_reward_learning-0.1.0-py3-none-any.whl size=16652 sha256=b28015a600073b24f6819b6bfc1a41ad215902712cbaaeb262059f2c5b001d06
  Stored in directory: /tmp/pip-ephem-wheel-cache-qrg1e_sd/wheels/c4/a3/46/a490b2cb8f10a30bfb4e1dc272885a0a4673033469e7092e3c
Successfully built hybrid_rnns_reward_learning
  Attempting uninstall: hybrid_rnns_reward_learning
    Found

In [2]:
#@title Import hybRNN functions
from hybrid_rnns_reward_learning import bi_rnn
from hybrid_rnns_reward_learning import cogmod
from hybrid_rnns_reward_learning import fit_hyb_rnn
from hybrid_rnns_reward_learning import hyb_rnn_utilities
from hybrid_rnns_reward_learning import rnn
from hybrid_rnns_reward_learning import rnn_config

In [3]:
#@title Download human dataset from OSF
import urllib
import os
import pandas as pd
import haiku as hk

download_url = 'https://osf.io/download/dw7f6/'
local_path = os.path.join('/tmp', 'openSourceRawDataset.csv')
urllib.request.urlretrieve(download_url, local_path)

data_df = pd.read_csv(local_path)
print(data_df.head())

   s_id  block  trial_id  action  reward     rt  payout_1  payout_2  payout_3  \
0   268    1.0         0     0.0    0.75  379.0      0.75      0.97      0.26   
1   268    1.0         1     2.0    0.28  212.0      0.74      0.99      0.28   
2   268    1.0         2     1.0    0.93  128.0      0.72      0.93      0.34   
3   268    1.0         3     3.0    0.47   68.0      0.73      0.98      0.22   
4   268    1.0         4     1.0    0.96  272.0      0.65      0.96      0.25   

   payout_4  
0      0.59  
1      0.53  
2      0.49  
3      0.47  
4      0.55  


# Set up training config

In [5]:
#@title Example 1: Fit "Best RL" model (free parameters: α, β, b, κ, Qinit, p)
config = rnn_config.get_config()
config.model_name = 'cogmod'
config.rnn_rl_params.fit_alpha = True
config.rnn_rl_params.fit_beta = True
config.rnn_rl_params.fit_bias = True
config.rnn_rl_params.fit_forget = True
config.rnn_rl_params.fit_init_v = True
config.rnn_rl_params.fit_persev_t = True
config.rnn_rl_params.fit_init_h = False
config.rnn_rl_params.fit_persev_p = False
config.rnn_rl_params.fit_w = False

In [6]:
#@title Example 2: Fit winning "hybRNN" model
config = rnn_config.get_config()
config.model_name = 'birnn'
config.rnn_rl_params.w_v = 1
config.rnn_rl_params.w_h = 1
config.rnn_rl_params.fit_forget = True
config.rnn_rl_params.o = False
config.rnn_rl_params.s = True
config.rnn_rl_params.zero_values = True
config.rnn_rl_params.fit_init_v = True
config.rnn_rl_params.fit_init_h = True

In [17]:
config.dataset_path = local_path
config.n_training_steps = 1001
config.batch_size = 32
# config

# Run the training loop

In [18]:
scalars, params = fit_hyb_rnn.train(config)

Loading data from /tmp/openSourceRawDataset.csv
Size of training data: 415 blocks.
Using BiRNN to fit data.
Start fitting the model
Step: 0,
Scalars: {'train_loss': [array(284.29706, dtype=float32)], 'step': [0], 'test_loss': [array(281.2657, dtype=float32)], 'valid_loss': [array(285.119, dtype=float32)]}
Step: 500,
Scalars: {'train_loss': [array(117.58526, dtype=float32)], 'step': [500], 'test_loss': [array(133.77505, dtype=float32)], 'valid_loss': [array(123.75827, dtype=float32)]}
Step: 1000,
Scalars: {'train_loss': [array(94.29534, dtype=float32)], 'step': [1000], 'test_loss': [array(113.82839, dtype=float32)], 'valid_loss': [array(104.73851, dtype=float32)]}


In [19]:
#@title Inspect the loss
scalars

{'train_loss': [array(94.29534, dtype=float32)],
 'step': [1000],
 'test_loss': [array(113.82839, dtype=float32)],
 'valid_loss': [array(104.73851, dtype=float32)]}

In [20]:
#@title Inspect fitted model parameters
params

{'bi_rnn': {'init_value_h': Array([-0.07311344], dtype=float32),
  'init_value_v': Array([0.68615806], dtype=float32),
  'unsigmoid_forget': Array([1.7564903], dtype=float32)},
 'bi_rnn/~_habit_rnn/linear': {'b': Array([ 0.01083845,  0.05495869,  0.03722108, -0.03754954, -0.00283628,
          0.00772736,  0.01782231,  0.04211747,  0.05468077, -0.03490454,
          0.01775484, -0.04196378, -0.0327814 , -0.01965697, -0.01454114,
          0.05213867], dtype=float32),
  'w': Array([[ 8.32769647e-02, -1.18925214e-01, -4.30995017e-01,
          -3.69875282e-01, -2.50002980e-01,  2.16721356e-01,
          -3.71438950e-01, -8.74657556e-02, -1.82552725e-01,
          -4.17605452e-02,  2.25944985e-02, -2.30584875e-01,
           2.34421398e-02, -3.71916569e-04,  8.24135542e-02,
           1.72779486e-01],
         [-1.76411673e-01,  3.28432739e-01,  3.94435585e-01,
           9.18600336e-02,  4.72009145e-02,  1.96796879e-01,
          -3.61832604e-02, -3.18898797e-01,  4.69833970e-01,
       